# AuralGuard — Kaggle Training Pipeline
Trains B1 → B2 → B3 → B5 → AuralGuard sequentially on Kaggle's free P100 GPU.

**Estimated total: ~14 hours.**
- B1 (LFCC-LCNN): ~30 min
- B2 (RawNet2): ~1 hr
- B3 (AASIST): ~2 hrs
- B5 (WavLM+AASIST): ~4 hrs
- AuralGuard (full): ~6 hrs

## 1. Setup
Downloads ASVspoof 2019 LA from Kaggle via API (requires internet).

In [ ]:
import os, shutil, sys, zipfile
from pathlib import Path

RAW = Path("/kaggle/working/data/raw/ASVspoof2019_LA")
KAGGLE_DIR = Path("/kaggle/working")

# ── Download LA.zip via Kaggle API ──
if not RAW.exists() or not any(RAW.iterdir()):
    print("Downloading ASVspoof 2019 LA from Kaggle (≈7.1 GB)...")
    from kaggle import KaggleApi
    api = KaggleApi()
    api.authenticate()
    # Download the dataset to /kaggle/working
    api.dataset_download_files(
        "awsaf49/asvpoof-2019-dataset",
        path=str(KAGGLE_DIR),
        unzip=True,
        quiet=False
    )
    print("Download complete!")
else:
    print("Dataset already present, skipping download.")

# ── Find and move LA partition ──
la_source = None
for candidate in [
    KAGGLE_DIR / "LA",
    KAGGLE_DIR / "asvpoof-2019-dataset" / "LA",
    KAGGLE_DIR / "asvspoof2019" / "LA",
]:
    if candidate.exists() and candidate.is_dir():
        la_source = candidate
        break

if la_source is None:
    # Search for any directory with ASVspoof2019_LA_train
    for d in KAGGLE_DIR.iterdir():
        if d.is_dir():
            for sub in d.iterdir():
                if sub.is_dir() and "ASVspoof2019_LA" in sub.name:
                    la_source = d
                    break

if la_source is None:
    raise RuntimeError("Could not find LA partition. Check Kaggle download.")

RAW.mkdir(parents=True, exist_ok=True)
for item in la_source.iterdir():
    dest = RAW / item.name
    if not dest.exists():
        if item.is_dir():
            shutil.copytree(item, dest)
        else:
            shutil.copy2(item, dest)

# Fix nested LA/LA/ structure if present
nested = RAW / "LA"
if nested.exists() and nested.is_dir():
    for item in nested.iterdir():
        shutil.move(str(item), str(RAW / item.name))
    nested.rmdir()

print("LA dataset ready at", RAW)
print("Contents:", [p.name for p in RAW.iterdir() if p.is_dir()])

In [ ]:
# ── Install AuralGuard ──
# Clone the repo
if not Path("/kaggle/working/auralguard").exists():
    !git clone https://github.com/MIHMahmudEli/auralguard.git /kaggle/working/auralguard

%cd /kaggle/working/auralguard

# Install deps (Kaggle already has PyTorch + CUDA)
!pip install -e .[train,dev] --quiet
!pip install datasets --quiet

print("Installation complete")

In [ ]:
# ── Build manifests ──
%cd /kaggle/working/auralguard

# Fix protocol suffix for train split
import pandas as pd
from pathlib import Path

RAW = Path("/kaggle/working/data/raw/ASVspoof2019_LA")
MANIFEST_DIR = Path("/kaggle/working/data/manifests")
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = ["utt_id", "path", "label", "attack", "dataset", "lang", "split", "codec"]

for split, suffix in [("train", "trn"), ("dev", "trl"), ("eval", "trl")]:
    proto = RAW / "ASVspoof2019_LA_cm_protocols" / f"ASVspoof2019.LA.cm.{split}.{suffix}.txt"
    audio_dir = RAW / f"ASVspoof2019_LA_{split}" / "flac"
    rows = []
    for line in proto.read_text().splitlines():
        parts = line.split()
        utt, attack, key = parts[1], parts[3], parts[4]
        rows.append({
            "utt_id": utt,
            "path": str(audio_dir / f"{utt}.flac"),
            "label": 0 if key == "bonafide" else 1,
            "attack": "bonafide" if key == "bonafide" else attack,
            "dataset": "asvspoof2019_la",
            "lang": "en",
            "split": split,
            "codec": "none",
        })
    df = pd.DataFrame(rows, columns=COLUMNS)
    out = MANIFEST_DIR / f"asvspoof2019_la_{split}.csv"
    df.to_csv(out, index=False)
    print(f"{out.name}: {len(df)} rows ({df.label.sum()} spoof)")

In [ ]:
# ── Build augmentation manifests (RIRs) ──
%cd /kaggle/working/auralguard

import pandas as pd
from pathlib import Path

MANIFEST_DIR = Path("/kaggle/working/data/manifests")
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

# RIRs (comes with the kaggle dataset: asvpoof-2019-dataset has RIRS_NOISES)
rir_root = Path("/kaggle/working/data/raw/RIRS_NOISES")
if not rir_root.exists():
    # Try alongside LA
    for p in Path("/kaggle/working").iterdir():
        if p.is_dir() and "RIR" in p.name.upper():
            rir_root = p
            break

if rir_root.exists():
    rows = []
    for rir_dir in ["real_rirs_isotropic_noises/real_rirs_isotropic_noises",
                    "simulated_rirs/mediumroom", "simulated_rirs/largeroom",
                    "simulated_rirs/smallroom"]:
        d = rir_root / rir_dir
        if d.exists():
            for f in sorted(d.rglob("*.wav")):
                rows.append({"path": str(f)})
    pd.DataFrame(rows).to_csv(MANIFEST_DIR / "rirs.csv", index=False)
    print(f"RIRs manifest: {len(rows)} files")
else:
    print("RIRs not found, reverb augmentation disabled")

# MUSAN not available on Kaggle, add_noise will be a no-op
pd.DataFrame(columns=["path"]).to_csv(MANIFEST_DIR / "musan.csv", index=False)
print("MUSAN manifest: empty (not available)")

## 2. Train Baselines
Each cell below trains one experiment. Run them sequentially.

**Tip:** After each DONE cell, check the loss curve in TensorBoard then move to the next.

In [ ]:
# ── B1: LFCC + Light CNN (≈30 min) ──
%cd /kaggle/working/auralguard
!python scripts/train.py experiment=b1_lcnn \
    data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv \
    data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv \
    data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv
print("B1 DONE")

In [ ]:
# ── B2: RawNet2 (≈1 hr) ──
%cd /kaggle/working/auralguard
!python scripts/train.py experiment=b2_rawnet2 \
    data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv \
    data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv \
    data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv
print("B2 DONE")

In [ ]:
# ── B3: AASIST (≈2 hrs) ──
%cd /kaggle/working/auralguard
!python scripts/train.py experiment=b3_aasist \
    data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv \
    data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv \
    data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv
print("B3 DONE")

In [ ]:
# ── B5: WavLM + AASIST + OC-Softmax (≈4 hrs) ──
%cd /kaggle/working/auralguard
!python scripts/train.py experiment=b5_wavlm_ocs \
    data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv \
    data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv \
    data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv
print("B5 DONE")

In [ ]:
# ── AuralGuard (≈6 hrs) ──
%cd /kaggle/working/auralguard
!python scripts/train.py experiment=auralguard \
    data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv \
    data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv \
    data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv
print("AURALGUARD DONE!")

## 3. Evaluate & Download Results
After training, evaluate all checkpoints and download CSV results back to your machine.

In [ ]:
# ── Evaluate all experiments ──
%cd /kaggle/working/auralguard

import pandas as pd
from pathlib import Path
import subprocess

results = []
for exp_dir in Path("experiments").iterdir():
    if not exp_dir.is_dir():
        continue
    ckpt = exp_dir / "checkpoints" / "best.ckpt"
    if ckpt.exists():
        print(f"Evaluating {exp_dir.name}...")
        subprocess.run([
            "python", "scripts/evaluate.py",
            f"{exp_dir.name}",
            f"--ckpt={ckpt}"
        ], check=True)

print("All evaluations complete!")

In [ ]:
# ── Package results for download ──
import shutil
from pathlib import Path

# Create a tarball of everything important
!tar czf /kaggle/working/auralguard_results.tar.gz \
    -C /kaggle/working/auralguard experiments/*/results.json \
    -C /kaggle/working/auralguard experiments/*/checkpoints/best.ckpt \
    -C /kaggle/working/auralguard experiments/*/logs/

print("Results packaged: /kaggle/working/auralguard_results.tar.gz")
print("Download via Kaggle sidebar → Output → Data")